# PetMind — YOLOv8 행동 인식 학습 (Google Colab)

> **실행 순서**: 위에서 아래로 셀을 하나씩 실행하세요.
> 
> **런타임 설정**: 런타임 → 런타임 유형 변경 → T4 GPU 선택

In [ ]:
# 1. GPU 확인
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
!nvidia-smi | head -20

In [ ]:
# 2. 패키지 설치
!pip install -q ultralytics roboflow python-dotenv

In [ ]:
# 3. Google Drive 마운트 (학습된 가중치 저장용)
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/PetMind'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Drive 마운트 완료:', SAVE_DIR)

In [ ]:
# 4. Roboflow에서 데이터셋 다운로드
# ⚠️ API 키를 입력하세요
ROBOFLOW_API_KEY = "anO75714mXs97va1Hnmt"  # .env 파일의 ROBOFLOW_API_KEY

from roboflow import Roboflow
import os

os.chdir('/content')

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# 포즈 데이터셋 (playing, resting, alert)
project = rf.workspace('dog-pose-annotation').project('dog-pose-feaal')
version = project.version(12)
dataset = version.download('yolov8', location='/content/pose_data')
print('포즈 데이터셋 다운로드 완료')

In [ ]:
# 5. 감정 데이터셋 다운로드 (happy, anxious용)
project2 = rf.workspace('dog-emotion-zaveh').project('dog-emotion-ovhny')
version2 = project2.version(2)
dataset2 = version2.download('yolov8', location='/content/emotion_data')
print('감정 데이터셋 다운로드 완료')

In [ ]:
# 6. 데이터 병합 및 전처리
import shutil
from pathlib import Path
import numpy as np
import cv2

# 클래스 인덱스
HAPPY_IDX = 0
ANXIOUS_IDX = 1

# 포즈 데이터 클래스 매핑
POSE_LABEL_MAP = {
    0: 2,  # playing (chien a pieds)
    1: 3,  # resting (chien assis)
    2: 4,  # alert (chien debout)
}

DATA_DIR = Path('/content/behavior_data')
for split in ['train', 'val', 'test']:
    (DATA_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATA_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

def remap_label_file(src_lbl, dst_lbl, label_map):
    lines = Path(src_lbl).read_text().strip().splitlines()
    new_lines = []
    for line in lines:
        parts = line.split()
        if not parts:
            continue
        cls = int(parts[0])
        if cls in label_map:
            parts[0] = str(label_map[cls])
            new_lines.append(' '.join(parts))
    if new_lines:
        Path(dst_lbl).write_text('\n'.join(new_lines))

# 포즈 데이터 복사 + 라벨 매핑
pose_base = Path('/content/pose_data')
for split in ['train', 'valid', 'test']:
    dst_split = 'val' if split == 'valid' else split
    img_dir = pose_base / split / 'images'
    lbl_dir = pose_base / split / 'labels'
    if not img_dir.exists():
        continue
    for img_path in img_dir.glob('*.*'):
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        dst_img = DATA_DIR / 'images' / dst_split / img_path.name
        dst_lbl = DATA_DIR / 'labels' / dst_split / (img_path.stem + '.txt')
        shutil.copy2(img_path, dst_img)
        if lbl_path.exists():
            remap_label_file(lbl_path, dst_lbl, POSE_LABEL_MAP)

print('포즈 데이터 병합 완료')

# 감정 데이터에서 happy, sad(anxious) 복사
emotion_base = Path('/content/emotion_data')

# 감정 클래스 이름 확인
import yaml
with open(emotion_base / 'data.yaml') as f:
    emotion_yaml = yaml.safe_load(f)
print('감정 클래스:', emotion_yaml.get('names'))

emotion_names = emotion_yaml.get('names', [])
# happy=0, sad=1 (또는 yaml 순서에 따라)
try:
    happy_src_idx = emotion_names.index('happy')
    sad_src_idx = emotion_names.index('sad')
except ValueError:
    happy_src_idx = 0
    sad_src_idx = 1

for split in ['train', 'valid', 'test']:
    dst_split = 'val' if split == 'valid' else split
    img_dir = emotion_base / split / 'images'
    lbl_dir = emotion_base / split / 'labels'
    if not img_dir.exists():
        continue
    for img_path in img_dir.glob('*.*'):
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if not lbl_path.exists():
            continue
        lines = lbl_path.read_text().strip().splitlines()
        # happy 또는 sad인 박스가 있는지 확인
        has_target = False
        new_lines = []
        for line in lines:
            parts = line.split()
            if not parts:
                continue
            cls = int(parts[0])
            if cls == happy_src_idx:
                parts[0] = str(HAPPY_IDX)
                new_lines.append(' '.join(parts))
                has_target = True
            elif cls == sad_src_idx:
                parts[0] = str(ANXIOUS_IDX)
                new_lines.append(' '.join(parts))
                has_target = True
        if has_target:
            dst_img = DATA_DIR / 'images' / dst_split / img_path.name
            dst_lbl = DATA_DIR / 'labels' / dst_split / (img_path.stem + '.txt')
            shutil.copy2(img_path, dst_img)
            dst_lbl.write_text('\n'.join(new_lines))

print('감정 데이터 병합 완료')

In [ ]:
# 7. 클래스 분포 확인
from collections import Counter
from pathlib import Path

LABELS = ['happy', 'anxious', 'playing', 'resting', 'alert']
DATA_DIR = Path('/content/behavior_data')

for split in ['train', 'val', 'test']:
    lbl_dir = DATA_DIR / 'labels' / split
    counts = Counter()
    for lbl_file in lbl_dir.glob('*.txt'):
        for line in lbl_file.read_text().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1
    print(f'\n[{split}]')
    for i, name in enumerate(LABELS):
        print(f'  {name:15s} {counts[i]:5d}')

In [ ]:
# 8. dataset.yaml 생성
import yaml
from pathlib import Path

dataset_cfg = {
    'path': '/content/behavior_data',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 5,
    'names': {0: 'happy', 1: 'anxious', 2: 'playing', 3: 'resting', 4: 'alert'}
}

with open('/content/dataset.yaml', 'w') as f:
    yaml.dump(dataset_cfg, f, allow_unicode=True)

print('dataset.yaml 생성 완료')
!cat /content/dataset.yaml

In [ ]:
# 9. YOLOv8 학습
from ultralytics import YOLO
import torch

print('CUDA:', torch.cuda.is_available())

model = YOLO('yolov8n.pt')
results = model.train(
    data='/content/dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=32,          # GPU 메모리에 따라 16~64 조정
    device='0',        # GPU 사용
    project='/content/drive/MyDrive/PetMind/weights',
    name='behavior_v1',
    patience=20,
    save=True,
    val=True,
    fl_gamma=1.5,      # 클래스 불균형 보정 (focal loss)
    cos_lr=True,
    seed=42,
    plots=True,
)

print('\n학습 완료!')
print(f'mAP50: {results.results_dict["metrics/mAP50(B)"]:.4f}')

In [ ]:
# 10. 학습 결과 확인
import os

weight_path = '/content/drive/MyDrive/PetMind/weights/behavior_v1/weights/best.pt'
if os.path.exists(weight_path):
    size_mb = os.path.getsize(weight_path) / 1024 / 1024
    print(f'best.pt 저장 완료: {size_mb:.1f} MB')
else:
    print('파일이 없습니다. 학습 중 오류가 발생했을 수 있습니다.')

# 검증 성능
from IPython.display import Image
results_img = '/content/drive/MyDrive/PetMind/weights/behavior_v1/results.png'
if os.path.exists(results_img):
    display(Image(results_img))